# Spiking Neural Networks (SNN)

## What is a Spiking Neural Network?

Standard **Artificial Neural Networks (ANNs)** pass continuous floating-point activations (e.g. `0.732`) between layers on every forward pass. **Spiking Neural Networks (SNNs)** — often called the "third generation" of neural networks — are biologically inspired: neurons only communicate via discrete **binary spikes** (0 or 1) emitted at specific points in time. Information is encoded in *when* and *how often* a neuron fires, not in the magnitude of a single number.

The most common neuron model is the **Leaky Integrate-and-Fire (LIF)** neuron:
- Each timestep, the membrane potential decays slightly (the *leak*)
- Incoming spikes add to the potential (weighted by synapse strength)
- If potential > threshold ⇒ **fire a spike**, then reset

$$
\textcolor{blue}{V(t+1) = \beta \cdot V(t) + \sum(w \cdot \text{input\_spikes}) - \text{threshold} \cdot \text{output\_spike}}
$$

β (typically 0.8–0.95) is the decay factor — higher β = longer memory.

---

### SNN vs. ANN — a side-by-side

| Aspect | Standard ANN | Spiking Neural Network |
|---|---|---|
| **Signal** | Continuous floats | Binary spikes (0/1) over time |
| **Time** | Single forward pass | Simulated over T timesteps |
| **Neuron** | ReLU / GELU / sigmoid | LIF (stateful, leaky integrator) |
| **Computation** | Dense MAC operations | Event-driven (only on spikes) |
| **Hardware fit** | GPU / TPU | Neuromorphic chips (Loihi, SpiNNaker, Akida) |
| **Energy** | Constant high cost per inference | Potentially **10–100× lower** on neuromorphic HW |
| **Training** | Direct backprop | Surrogate gradients (spike step is non-differentiable) |

### ✅ Advantages of SNNs
- **Energy efficiency** — sparse, event-driven activity means most neurons stay silent most of the time. Ideal for always-on edge devices, IoT sensors, wearables.
- **Native temporal processing** — built-in notion of time makes them well-suited for audio, event-camera (DVS) data, and sensor streams.
- **Biological plausibility** — closer to how real brains compute, useful for neuroscience research.
- **Neuromorphic hardware acceleration** — chips like Intel Loihi 2 can run SNNs at milliwatt power budgets.

### ⚠️ Drawbacks
- **Harder to train** — the spike function has zero gradient almost everywhere; we need *surrogate gradients* as a workaround.
- **Slower simulation on GPUs** — looping over T timesteps removes the parallelism advantage GPUs are built for.
- **Lower accuracy ceiling** (currently) — SNNs typically trail equivalent ANNs by a few percentage points on standard benchmarks.
- **Tooling immaturity** — fewer libraries, pretrained models, and deployment options compared to PyTorch/TensorFlow ecosystems.
- **Encoding overhead** — static inputs (like an image) must be converted into spike trains via rate / latency / population coding, adding a design choice.

---

In this notebook we train a **3-layer LIF-based SNN** on MNIST using [snnTorch](https://snntorch.readthedocs.io/), with **rate coding** to convert pixel intensities into spike trains over `T=25` timesteps.

In [1]:
import torch
import torch.nn as nn
from  torchvision import datasets, transforms
from torch.utils.data import DataLoader

import snntorch as snn
from snntorch import functional as SF
from snntorch import spikegen

In [2]:
BATCH_SIZE = 128
EPOCHS=15
LR=1e-3
NUM_TIMESTEPS=25 # How many time steps to simulate (more is better accuracy but slower)
BETA=0.9 # membrane potential decay rate (0 = no memory, 1 = no leak)
NUM_CLASSES=10
DEVICE =torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

Using Device: cuda


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(), # converts to [0,1] range
])

train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("./data",train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

In [4]:
class SpikingNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # synaptic weights (same as regular linear layers)
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, NUM_CLASSES)
        
        # LIF neurons - one per layer
        # beta - decay rate of membrane potential each timestep
        # Internally handles: membrane update, threshold comparison, spike generation, reset
        self.lif1 = snn.Leaky(beta=BETA)
        self.lif2 = snn.Leaky(beta=BETA)
        self.lif3 = snn.Leaky(beta=BETA)
        
    def forward(self, x):
        # --- rate coding: convert pixel intensities to spike trains ---
        # A pixel with value 0.8 will spike with a probability of 0.8 each time step.
        # shape goes from (batch, 1, 28, 28) => (T, batch, 1, 28, 28)
        spike_input = spikegen.rate(x, num_steps=NUM_TIMESTEPS)
        
        # Flatten spatial dimensions (T, batch, 1, 28, 28) => (T, batch, 784)
        spike_input = spike_input.view(NUM_TIMESTEPS, x.size(0), -1)
        
        # initialize membrane potentials to zero
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        
        # collect output spikes and membrane potentials over time
        spike_record = []
        mem_record = []
        
        for t in range(NUM_TIMESTEPS):
            # Layer 1 : linear transform => LIF neuron
            cur1 = self.fc1(spike_input[t]) # synaptic current
            spk1, mem1 = self.lif1(cur1, mem1) # spike + updated membrane
            
            # Layer 2
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            # Layer 3 (output layer)
            cur3 = self.fc3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)
            
            spike_record.append(spk3)
            mem_record.append(mem3)
                
        return torch.stack(spike_record), torch.stack(mem_record)

In [5]:
model = SpikingNet().to(DEVICE)
print(f"model {model}")

model SpikingNet(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
  (lif1): Leaky()
  (lif2): Leaky()
  (lif3): Leaky()
)


### Loss function optimizer
* We use cross entropy loss on the total output spike count over all timesteps.
* The class with the most spikes wins
* Alternatively: use the final membrane potential (mem_record[-1]) - sometimes works better.

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = SF.ce_count_loss() # cross entropy on spike counts

In [7]:
# Training loop
def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        # Forward pass: simulate the SNN over T time steps
        spike_record, mem_record = model(images)
        
        # Loss: Cross-Entropy on total spike counts per class
        loss = loss_fn(spike_record, labels)
        
        # Backward pass: surrogate gradient are automatically applied by snnTorch
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # track accuracy
        total_spikes = spike_record.sum(dim=0) # (batch, num classes)
        predicted = total_spikes.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"  Epoch {epoch} | Batch {batch_idx}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}")
 
    avg_loss = total_loss / len(train_loader)
    accuracy = 100.0 * correct / total
    print(f"  Epoch {epoch} Train — Loss: {avg_loss:.4f}, Accuracy: {accuracy:.1f}%")
    return avg_loss, accuracy
        

In [8]:
def evaluate():
    model.eval()
    correct = 0
    total = 0
 
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            spike_record, _ = model(images)
            total_spikes = spike_record.sum(dim=0)
            predicted = total_spikes.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
 
    accuracy = 100.0 * correct / total
    print(f"  Test Accuracy: {accuracy:.1f}%\n")
    return accuracy

In [9]:
print("=" * 60)
print("Starting SNN Training on MNIST")
print(f"Timesteps: {NUM_TIMESTEPS} | Beta: {BETA} | LR: {LR}")
print("=" * 60)
 
best_acc = 0
for epoch in range(1, EPOCHS + 1):
    train_one_epoch(epoch)
    acc = evaluate()
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "snn_mnist_best.pt")
        print(f"  ✓ Saved best model (accuracy: {best_acc:.1f}%)")
 
print("=" * 60)
print(f"Training complete! Best test accuracy: {best_acc:.1f}%")
print("Model saved to: snn_mnist_best.pt")
print("=" * 60)

Starting SNN Training on MNIST
Timesteps: 25 | Beta: 0.9 | LR: 0.001
  Epoch 1 | Batch 0/468 | Loss: 2.3026
  Epoch 1 | Batch 100/468 | Loss: 0.3333
  Epoch 1 | Batch 200/468 | Loss: 0.3031
  Epoch 1 | Batch 300/468 | Loss: 0.1096
  Epoch 1 | Batch 400/468 | Loss: 0.1170
  Epoch 1 Train — Loss: 0.2553, Accuracy: 92.0%
  Test Accuracy: 96.0%

  ✓ Saved best model (accuracy: 96.0%)
  Epoch 2 | Batch 0/468 | Loss: 0.1586
  Epoch 2 | Batch 100/468 | Loss: 0.1443
  Epoch 2 | Batch 200/468 | Loss: 0.1641
  Epoch 2 | Batch 300/468 | Loss: 0.0780
  Epoch 2 | Batch 400/468 | Loss: 0.1195
  Epoch 2 Train — Loss: 0.0931, Accuracy: 97.1%
  Test Accuracy: 97.1%

  ✓ Saved best model (accuracy: 97.1%)
  Epoch 3 | Batch 0/468 | Loss: 0.0668
  Epoch 3 | Batch 100/468 | Loss: 0.0958
  Epoch 3 | Batch 200/468 | Loss: 0.0427
  Epoch 3 | Batch 300/468 | Loss: 0.0262
  Epoch 3 | Batch 400/468 | Loss: 0.0445
  Epoch 3 Train — Loss: 0.0652, Accuracy: 97.9%
  Test Accuracy: 97.4%

  ✓ Saved best model (accura

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
})

def run_inference_and_visualize(n_samples=10):
    model.eval()

    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    images = images[:n_samples].to(DEVICE)
    labels = labels[:n_samples].to(DEVICE)

    with torch.no_grad():
        spike_record, mem_record = model(images)
        total_spikes = spike_record.sum(dim=0)            # (B, 10)
        predicted = total_spikes.argmax(dim=1)
        confidence = torch.softmax(total_spikes, dim=1).max(dim=1).values

    images_cpu = images.cpu()
    labels_cpu = labels.cpu()
    predicted_cpu = predicted.cpu()
    conf_cpu = confidence.cpu()

    fig, axes = plt.subplots(1, n_samples, figsize=(2 * n_samples, 2.6))
    fig.suptitle(
        f"SNN on MNIST — {NUM_TIMESTEPS} timesteps, β={BETA}",
        fontsize=14, fontweight="bold", y=1.05,
    )

    for i, ax in enumerate(axes):
        ax.imshow(images_cpu[i].squeeze(), cmap="gray")
        correct = predicted_cpu[i].item() == labels_cpu[i].item()
        color = "#2ecc71" if correct else "#e74c3c"
        mark = "✓" if correct else "✗"
        ax.set_title(
            f"{mark} pred: {predicted_cpu[i].item()}  (true: {labels_cpu[i].item()})\n"
            f"conf: {conf_cpu[i].item()*100:.1f}%",
            fontsize=10, color=color, fontweight="bold",
        )
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2.5)
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    plt.savefig("snn_predictions.png", dpi=150, bbox_inches="tight")
    plt.show()

    return spike_record, mem_record, images, labels, predicted

In [ ]:
model.load_state_dict(torch.load("snn_mnist_best.pt"))
spike_record, mem_record, images, labels, predicted = run_inference_and_visualize()

### Peeking inside the SNN — spike rasters & membrane potentials

For one test sample, we visualize:
1. The **input spike raster** — pixels rate-coded into binary spike trains over time.
2. The **output spike raster** — when each of the 10 output neurons fires.
3. The **membrane potentials** — how each output neuron integrates evidence over the 25 timesteps. The neuron whose potential climbs fastest "wins".

In [ ]:
def visualize_spike_dynamics(sample_idx=0):
    img = images[sample_idx:sample_idx+1]
    true_label = labels[sample_idx].item()
    pred_label = predicted[sample_idx].item()

    # Re-encode input to get the input spike train
    with torch.no_grad():
        in_spikes = spikegen.rate(img, num_steps=NUM_TIMESTEPS)         # (T,1,1,28,28)
        in_spikes_flat = in_spikes.view(NUM_TIMESTEPS, -1).cpu().numpy() # (T, 784)

    out_spikes = spike_record[:, sample_idx, :].cpu().numpy()  # (T, 10)
    out_mem    = mem_record[:, sample_idx, :].cpu().numpy()    # (T, 10)

    fig = plt.figure(figsize=(16, 6), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 1.3, 1.3])

    # 1. Input image
    ax0 = fig.add_subplot(gs[:, 0])
    ax0.imshow(img.cpu().squeeze(), cmap="gray")
    ax0.set_title(f"Input digit\nTrue: {true_label} | Pred: {pred_label}",
                  fontsize=13, fontweight="bold")
    ax0.axis("off")

    # 2. Input spike raster (subset of pixels for clarity)
    ax1 = fig.add_subplot(gs[0, 1])
    active_pixels = np.where(in_spikes_flat.sum(axis=0) > 0)[0]
    sub = in_spikes_flat[:, active_pixels[:120]]
    t_idx, p_idx = np.where(sub.T > 0)
    ax1.scatter(p_idx, t_idx, s=2, c="#3498db", marker="|")
    ax1.set_xlabel("Timestep"); ax1.set_ylabel("Pixel #")
    ax1.set_title("Input spike raster (rate-coded pixels)", fontweight="bold")
    ax1.set_xlim(0, NUM_TIMESTEPS)

    # 3. Output spike raster
    ax2 = fig.add_subplot(gs[1, 1])
    t_o, c_o = np.where(out_spikes > 0)
    colors = ["#e74c3c" if c == pred_label else "#7f8c8d" for c in c_o]
    ax2.scatter(t_o, c_o, s=80, c=colors, marker="|", linewidths=2)
    ax2.set_yticks(range(10))
    ax2.set_xlabel("Timestep"); ax2.set_ylabel("Output neuron (class)")
    ax2.set_title("Output spike raster (red = predicted class)", fontweight="bold")
    ax2.set_xlim(-0.5, NUM_TIMESTEPS - 0.5)
    ax2.set_ylim(-0.5, 9.5)
    ax2.grid(alpha=0.3)

    # 4. Membrane potentials over time
    ax3 = fig.add_subplot(gs[:, 2])
    cmap = plt.cm.tab10
    for c in range(10):
        lw = 2.8 if c == pred_label else 1.0
        alpha = 1.0 if c == pred_label else 0.5
        ax3.plot(out_mem[:, c], label=f"class {c}", color=cmap(c),
                 linewidth=lw, alpha=alpha)
    ax3.set_xlabel("Timestep"); ax3.set_ylabel("Membrane potential V(t)")
    ax3.set_title("Output-layer membrane potentials\n(thick red-ish = winning class)",
                  fontweight="bold")
    ax3.legend(loc="upper left", fontsize=8, ncol=2)
    ax3.grid(alpha=0.3)

    plt.savefig("snn_dynamics.png", dpi=150, bbox_inches="tight")
    plt.show()

visualize_spike_dynamics(sample_idx=0)

### Training curve & confusion matrix
A quick look at how the SNN learned over 15 epochs, and where it still confuses digits.

In [ ]:
# Training history captured from the run above
train_acc = [92.0, 97.1, 97.9, 98.4, 98.7, 99.0, 99.1, 99.1, 99.2, 99.3, 99.4, 99.4, 99.4, 99.4, 99.5]
test_acc  = [96.0, 97.1, 97.4, 97.8, 97.6, 97.6, 97.5, 97.9, 97.8, 97.5, 97.6, 97.9, 97.7, 98.0, 97.6]
train_loss = [0.2553, 0.0931, 0.0652, 0.0494, 0.0389, 0.0309, 0.0280, 0.0260,
              0.0234, 0.0217, 0.0193, 0.0163, 0.0171, 0.0161, 0.0143]
epochs = list(range(1, len(train_acc) + 1))

# --- Build full confusion matrix on the test set ---
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE)
        sp, _ = model(imgs)
        all_preds.append(sp.sum(dim=0).argmax(dim=1).cpu())
        all_labels.append(lbls)
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = np.zeros((10, 10), dtype=int)
for t, p in zip(all_labels, all_preds):
    cm[t, p] += 1
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# (a) Accuracy / loss curves
ax = axes[0]
ax.plot(epochs, train_acc, "o-", color="#2980b9", label="Train accuracy", linewidth=2)
ax.plot(epochs, test_acc,  "s-", color="#27ae60", label="Test accuracy",  linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
ax.set_title("SNN learning curve on MNIST", fontweight="bold", fontsize=13)
ax.set_ylim(90, 100)
ax.grid(alpha=0.3); ax.legend(loc="lower right")
ax.axhline(98.0, color="gray", linestyle="--", alpha=0.6)
ax.text(15, 98.1, "best test: 98.0%", fontsize=9, ha="right", color="gray")

ax_l = ax.twinx()
ax_l.plot(epochs, train_loss, "^--", color="#e67e22", alpha=0.6, label="Train loss")
ax_l.set_ylabel("Loss", color="#e67e22")
ax_l.tick_params(axis="y", labelcolor="#e67e22")

# (b) Confusion matrix
ax = axes[1]
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized)", fontweight="bold", fontsize=13)
for i in range(10):
    for j in range(10):
        v = cm_norm[i, j]
        if v > 0.01:
            ax.text(j, i, f"{v*100:.0f}", ha="center", va="center",
                    color="white" if v > 0.5 else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig("snn_training.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Final test accuracy: {(all_preds == all_labels).mean()*100:.2f}%")

## Takeaways

- A simple 3-layer LIF spiking network reaches **98.0% test accuracy** on MNIST — within ~1% of an equivalent dense ANN.
- Inference is fully **event-driven**: the prediction emerges from *spike counts* and *membrane integration over time*, not a single forward pass.
- The same model on **neuromorphic hardware** (Intel Loihi 2, BrainChip Akida) would run at a fraction of the energy of a GPU.

### Where SNNs shine
- Always-on edge inference (keyword spotting, anomaly detection)
- Event-camera (DVS) vision and high-speed robotics
- Brain–machine interfaces and biologically faithful modeling

### What's still hard
- Training is slower and noisier than ANNs because of surrogate gradients.
- GPU simulation negates the energy advantage — you really need neuromorphic silicon to feel the win.
- Static benchmarks (MNIST, CIFAR) underplay SNN strengths; their real edge is on **temporal / sparse** data.

> *Built with [snnTorch](https://snntorch.readthedocs.io/) + PyTorch.*